# പാഠം 01 - AI ഏജന്റുകളിലേക്ക് പരിചയം

**AI ഏജന്റുകൾ തുടങ്ങുന്നവർക്ക്** കോഴ്സിലെ ആദ്യ പാഠത്തിലേക്ക് സ്വാഗതം!

ഒരു **AI ഏজന്റ്** എന്നത് ഒരു വലിയ ഭാഷാ മോഡൽ (LLM) അതിന്റെ ബോധവല്‍ക്കരണ എൻജിനായി ഉപയോഗിച്ച്, യഥാർത്ഥ ലോകത്തിൽ *പ്രവർത്തനങ്ങൾ* നടത്താവുന്ന — API കളിൽ വിളിക്കലുകൾ, ഡാറ്റാബേസുകൾ പരിശോധിക്കൽ, അല്ലെങ്കിൽ കോഡ് നിർവഹിക്കൽ — ഒരു ഉപയോക്താവിന്റെ വേണ്ടി ഒരു ലക്ഷ്യം കൈവരിക്കാൻ സഹായിക്കുന്ന പ്രോഗ്രാമാണ്.

ഈ നോട്ട്ബുക്കിൽ നിങ്ങൾ നിങ്ങളുടെ ആദ്യ ഏജന്റ് നിർമ്മിക്കും: അവധി കേന്ദ്രങ്ങൾ ശിപാർശ ചെയ്യുന്ന ഒരു **ട്രാവൽ ഏജന്റ്**. ഇതിന്റെ വഴിയിൽ നിങ്ങൾക്കു പഠിക്കാം:

1. **Microsoft Agent Framework** ഉപയോഗിച്ച് Azure AI Foundry Agent സേവനത്തിലേക്ക് ബന്ധപ്പെടുന്നത്.
2. ഏജന്റിന് ഒരു **ടൂൾ** നൽകുക — അത് വിളിക്കാൻ കഴിയുന്ന ഒരു സാധാരണ പൈതൺ ഫങ്‌ഷൻ.
3. ഏജന്റ് പ്രവർത്തിപ്പിച്ച് അതിന്റെ പ്രതികരണം പരിശോധിക്കുക.
4. ഏജന്റിന്റെ പ്രതികരണം ടോക്കൺ-ടോക്കൺ ആയി സ്റ്റ്രീം ചെയ്യുക.


## സജ്ജീകരണം

ഈ നോട്ട്‌ബുക്ക്‌ ഓടിക്കുന്നതിന് മുമ്പ്, നിങ്ങൾക്കുണ്ട് എന്ന് ഉറപ്പാക്കുക:

1. **പ്രവർത്തനവിധേയമാക്കിയ ഒരു ചാറ്റ് മോഡലുള്ള Azure AI Foundry പ്രോജക്ട്** (മാതൃകയ്ക്ക് `gpt-4o-mini`).
2. **Azure CLI ഉപയോഗിച്ച് പ്രവേശനം നടത്തിയിട്ടുള്ളത്** — ടെർമിനലിൽ `az login` നടത്തുക.
3. **ആവശ്യമുള്ള പരിസ്ഥിതി വ്യത്യാസങ്ങൾ സജ്ജീകരിച്ചിട്ടുള്ളത്:**
   - `AZURE_AI_PROJECT_ENDPOINT` — നിങ്ങളുടെ Azure AI Foundry പ്രോജക്ട് എൻഡ്പോയിന്റ്.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — പ്രവർത്തനവിധേയമാക്കിയ മോഡലിന്റെ പേര്.

താഴെയിടുന്ന സെൽ നിങ്ങൾക്ക് ആവശ്യമായ Python പാക്കേജുകൾ ഇൻസ്റ്റാൾ ചെയ്യും.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## നിങ്ങളുടെ ആദ്യ ഏജന്റ് സൃഷ്ടിക്കുന്നത്

ഒരു ഏജന്റിന് രണ്ടു കാര്യങ്ങൾ ആവശ്യമുണ്ട്:

- അതിന് *ആരാണ്* എന്നും *എങ്ങനെ പെരുമരുത്തണം* എന്നും പറയും **നിർദ്ദേശങ്ങൾ** (ഒരു സിസ്റ്റം പ്രോംപ്റ്റ്).
- ഏജന്റ് വിവരങ്ങൾ ശേഖരിക്കാനോ പ്രവർത്തനങ്ങൾ നിർവഹിക്കാനോ വിളിക്കാവുന്ന `@tool` ഉപയോഗിച്ചുള്ള Python ഫังก്ഷനുകൾ — **ഉപകരണങ്ങൾ**.

താഴെ, പ്രചാരത്തിലുള്ള അവധിക്കാല യാത്രാ കേന്ദ്രങ്ങളുടെ ലിസ്റ്റ് നൽകുന്ന ഒരു ലളിതമായ ഉപകരണം നിർവചിക്കുന്നു. ഉപയോക്താവ് യാത്രാ ശിപാർശകൾ ആവശ്യപ്പെടുമ്പോൾ ഏജന്റ് ഈ ഉപകരണം ഉപയോഗിക്കും.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

അധിക സംവാദപരമായ അനുഭവത്തിന് നിങ്ങൾ ആജന്റിന്റെ പ്രതികരണം **_Stream_** ചെയാം. പൂർണ്ണ മറുപടി കാത്തിരിക്കാൻ പകരം, ആജന്റ് സൃഷ്ടിക്കുന്നതുപോലെ ടെക്സ്റ്റ് ഘടകങ്ങൾ നൽകുന്നു. ഇത് പ്രത്യേകിച്ച് ചാറ്റ് ഇന്റർഫേസുകളിൽ ഉപയോഗപ്രദമാണ്, ഇവിടെ നിങ്ങൾക്ക് സമയം ഞേടെ ഔട്പുട്ട് പ്രദർശിപ്പിക്കാം.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## സംഗ്രഹം

ഈ പാഠത്തിൽ നിങ്ങൾ എങ്ങനെ ചെയ്യാമെന്ന് പഠിച്ചു:

- **ഒരു പ്രൊവൈഡർ സൃഷ്ടിക്കുക** `AzureAIProjectAgentProvider` മുഖേന Azure AI Foundry Agent Service-യ്ക്ക് ബന്ധിപ്പിക്കാൻ.
- **ഒരു ടൂൾ നിർവചിക്കുക** `@tool` ഡെക്കറേറ്റർ ഉപയോഗിച്ച്, അങ്ങനെ ഏജന്റ് നിങ്ങളുടെ Python ഫംഗ്ഷനുകൾ വിളിക്കാൻ കഴിയും.
- **ഏജന്റ് റൺ ചെയ്യുക** ഒരു ഉപയോക്തൃ സന്ദേശത്തോടൊപ്പം, അതിന്റെ ഉത്തരം പ്രിന്റ് ചെയ്യുക.
- **സ്ട്രീം ചെയ്യുക പ്രതികരണങ്ങൾ** റിയൽ-ടൈം ഔട്ട്പുട്ടിനായി.

അടുത്ത പാഠത്തിൽ, ഏജന്റിക് ഫ്രെയിംവർക്കുകളെ കൂടുതൽ ആഴത്തിൽ പരിശോധിച്ച് ഏജന്റുകൾക്ക് കൂടുതൽ ശക്തമായ ടൂളുകളും multi-step reasoning ശേഷികളും നൽകാൻ എങ്ങനെ എന്നതിൽ പഠിക്കും.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അസ്വീകാരം**:  
ഈ പ്രമാണം AI പരിഭാഷ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്._accuracy നിഷ്‌പ്രയത്നം ഉറപ്പുവരുത്തുന്നതിന് ശ്രമിക്കുന്നിരുന്നാലും, യാന്ത്രിക പരിഭാഷകളിൽ പിശകുകൾ ഉണ്ടാകുന്നതിനും തെറ്റുകൾ സംഭവിക്കാനുള്ള സാധ്യത ഉണ്ടെന്ന് தயവായി ശ്രദ്ധിക്കുക. യഥാർത്ഥ പ്രമാണം അതിന്റെ സ്വദേശ ഭാഷയിൽ ന്യായാധിപത്യ സ്രോതസ്സായി പരിഗണിക്കപ്പെടണം. നിർണായക വിവരങ്ങൾക്കായി വിദഗ്ധ മനുഷ്യ പരിഭാഷ പ്രാധാന്യമുള്ളതാണ്. ഈ പരിഭാഷയിലെ ഉപയോക്തൃ ഉപയോഗത്തിൽ നിന്നുണ്ടാകുന്ന വാചകാര്‍ഥബോധമില്ലായ്മകൾക്ക് അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്ക് ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
